In [ ]:
from google.colab import drive
drive.mount('/content/gdrive/')

In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece protobuf latex2sympy2 sympy

In [ ]:
import os, sys, re, time, torch
import sympy as sp
from sympy import symbols, simplify, N

BASE_DIR    = '/content/gdrive/MyDrive/NLP_assignment'
PACKAGE_DIR = os.path.join(BASE_DIR, 'millionaire_client')

if not os.path.exists(BASE_DIR):
    print(f"Error path {BASE_DIR} not found. Please check your Google Drive paths.")

sys.path.append(BASE_DIR)
print("Environment ready")


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch, gc

MODEL_ID = "Qwen/Qwen2.5-Math-1.5B-Instruct"
PLANNER_ID = "Qwen/Qwen2.5-7B-Instruct"

print("Loading JSON planner Qwen in four bit mode")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

planner_tokenizer = AutoTokenizer.from_pretrained(PLANNER_ID)
if planner_tokenizer.pad_token is None:
    planner_tokenizer.pad_token = planner_tokenizer.eos_token

planner_model = AutoModelForCausalLM.from_pretrained(
    PLANNER_ID,
    quantization_config=bnb_config,
    device_map="auto"
).eval()

print(f"Tool planner loaded {PLANNER_ID}")
if torch.cuda.is_available():
    print(f"GPU memory allocated {torch.cuda.memory_allocated() / 1e9:.2f} GB")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Loading Math model Qwen without quantization")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16
).eval()

print(f"Math model loaded {MODEL_ID}")
if torch.cuda.is_available():
    print(f"GPU memory allocated {torch.cuda.memory_allocated() / 1e9:.2f} GB")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# Change this before starting the game.
# True uses the preloaded Qwen planner and calculator tools.
# False skips the toolbox and answers directly with Qwen Math.
USE_TOOL = True

print(f"Use tool is {USE_TOOL}")
print(f"Planner model is {PLANNER_ID}")


In [ ]:
import re, time, json, math, torch
from transformers import StoppingCriteria, StoppingCriteriaList
import sympy as sp
from fractions import Fraction
from sympy import N, symbols, simplify
from sympy.parsing.sympy_parser import parse_expr, standard_transformations, implicit_multiplication_application, convert_xor

TRANSFORMS = standard_transformations + (implicit_multiplication_application, convert_xor)

class TimeLimitStoppingCriteria(StoppingCriteria):
    def __init__(self, start_time: float, time_limit: float):
        self.start_time = start_time
        self.time_limit = time_limit

    def __call__(self, input_ids, scores, **kwargs):
        return time.time() - self.start_time >= self.time_limit
SYM_LOCALS = {'x': sp.symbols('x'), 'y': sp.symbols('y'), 'z': sp.symbols('z'), 'a': sp.symbols('a'), 'b': sp.symbols('b'), 'k': sp.symbols('k'), 't': sp.symbols('t'), 'n': sp.symbols('n'), 'pi': sp.pi, 'e': sp.E, 'E': sp.E}

TOOL_DESCRIPTIONS = '''
You are a ROUTER, not a solver. Your job is to choose a tool or no tool.
You may think briefly, but do not solve the full problem or choose the final option.
After thinking, output FINAL_JSON: followed by exactly ONE minified JSON object.
The JSON object must have no duplicate keys and no text after it.

Tools:
- sympy_simplify: simplify/evaluate expressions, radicals, powers, fractions, rational denominator.
- sympy_solve: equations, systems, roots, root sums/products/differences.
- sympy_calculus: derivative, integral, limit, critical points.
- sympy_matrix: characteristic polynomial, eigenvalues, trace, determinant.
- probability_stats: expected value, binomial, hypergeometric/draws without replacement, Bayes, normal quartile sigma, r^2, combinations, broken-stick triangle probability.
- none: conceptual/theorem/topology/group/sampling questions with no computation tool needed.

Output schemas only:
- No tool: {"action":"none"}
- Simplify tool: {"action":"tool","tool":"sympy_simplify","input":{"latex":REAL_LATEX_OR_EMPTY,"expression":REAL_EXPRESSION_OR_EMPTY}}
- Equation tool: {"action":"tool","tool":"sympy_solve","input":{"equations":[REAL_EQUATION_STRINGS],"variables":[REAL_VARIABLES],"target":REAL_TARGET}}
- Calculus tool: {"action":"tool","tool":"sympy_calculus","input":{"operation":REAL_OPERATION,"expression":REAL_EXPRESSION,"variable":REAL_VARIABLE}}
- Matrix tool: {"action":"tool","tool":"sympy_matrix","input":{"operation":REAL_OPERATION,"polynomial":REAL_POLYNOMIAL,"variable":REAL_VARIABLE}}
- Probability/statistics tool: {"action":"tool","tool":"probability_stats","input":{"operation":REAL_OPERATION,"values":REAL_VALUES_OBJECT}}

Use only values literally present in the real question, or values directly named by the real question.
For probability/chance/random/selected/exactly/at least/without replacement questions, prefer probability_stats.
For simplify/evaluate/radical/fraction/rational-denominator questions, prefer sympy_simplify.
If the question is only a conceptual rule/theorem/definition question, output {"action":"none"}.
If you are unsure how to build a valid tool input, output {"action":"none"}.
Format:
THINK: one short sentence about whether a tool is useful.
FINAL_JSON: {"action":"none"} OR {"action":"tool",...}
'''

def normalize_math_text(s: str) -> str:
    s = str(s).replace('^', '**').replace(chr(8722), '-').replace(chr(8211), '-').replace(chr(955), 'l')
    s = re.sub(r'(?<![A-Za-z])e\s*\^', 'E^', s)
    return s

def parse_math_expr(s: str, local_dict=None):
    local = dict(SYM_LOCALS)
    if local_dict: local.update(local_dict)
    return parse_expr(normalize_math_text(s), transformations=TRANSFORMS, local_dict=local)

def parse_json_object(raw: str):
    start = raw.find('{')
    if start < 0: return None
    depth = 0
    in_str = False
    escape = False
    for i, ch in enumerate(raw[start:], start):
        if in_str:
            if escape:
                escape = False
            elif ch == '\\':
                escape = True
            elif ch == '"':
                in_str = False
        else:
            if ch == '"': in_str = True
            elif ch == '{': depth += 1
            elif ch == '}':
                depth -= 1
                if depth == 0:
                    try: return json.loads(raw[start:i+1])
                    except Exception: return None
    return None

def parse_tool_plan(raw: str):
    if 'FINAL_JSON:' in raw:
        raw = raw.split('FINAL_JSON:', 1)[1]
    parsed = parse_json_object(raw)
    if parsed:
        return parsed

    def grab_string(key):
        m = re.search(r'"\s*' + re.escape(key) + r'\s*"\s*:\s*"([^"]*)"', raw, re.IGNORECASE)
        return m.group(1).strip() if m else ''

    action = grab_string('action')
    tool = grab_string('tool')
    if not action and re.search(r'"action"\s*:\s*"?\s*tool', raw, re.IGNORECASE):
        action = 'tool'
    if not action and re.search(r'"action"\s*:\s*"?\s*none', raw, re.IGNORECASE):
        action = 'none'
    if not tool:
        m_tool = re.search(r'"\s*tool\s*"\s*:\s*"?\s*([A-Za-z_.]+)', raw, re.IGNORECASE)
        if m_tool: tool = m_tool.group(1).strip()

    payload = {}
    eq_m = re.search(r'"equations"\s*:\s*\[([^\]]*)\]', raw, re.IGNORECASE | re.S)
    if eq_m:
        payload['equations'] = re.findall(r'"([^"]+)"', eq_m.group(1))
    vars_m = re.search(r'"variables"\s*:\s*\[([^\]]*)\]', raw, re.IGNORECASE | re.S)
    if vars_m:
        payload['variables'] = re.findall(r'"([^"]+)"', vars_m.group(1))
    for key in ['operation', 'expression', 'latex', 'variable', 'point', 'target', 'polynomial']:
        val = grab_string(key)
        if val: payload[key] = val

    if action or tool or payload:
        return {'action': action or 'tool', 'tool': tool, 'input': payload}
    return None

def llm_chat(tokenizer, model, messages, max_new_tokens=120, max_time=None, assistant_prefix=''):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    if assistant_prefix:
        prompt += assistant_prefix
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    input_len = inputs['input_ids'].shape[1]
    kwargs = dict(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    if max_time is not None: kwargs['max_time'] = max_time
    with torch.no_grad():
        outputs = model.generate(**kwargs)
    return assistant_prefix + tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

def extract_option_letter(raw: str):
    boxed = re.findall(r'\\boxed\{([A-D])\}', raw, re.IGNORECASE)
    if boxed: return boxed[-1].upper(), 'boxed'
    m = re.search(r'(?:FINAL ANSWER|answer is|answer|option|choice)\s*:?\s*([A-D])\b', raw, re.IGNORECASE)
    if m: return m.group(1).upper(), 'text_match'
    letters = re.findall(r'\b([A-D])\b', raw.upper())
    return (letters[-1], 'fallback') if letters else ('C', 'fallback')

def extract_answer_letter(raw: str):
    boxed = re.findall(r'\\boxed\{\s*([A-D])\s*\}', raw, re.IGNORECASE)
    if boxed:
        return boxed[-1].upper()
    m = re.search(r'(?i)\b(?:final answer|answer is|answer|option|choice)\s*[:\-]?\s*([A-D])\b', raw)
    if m:
        return m.group(1).upper()
    return None

def run_sympy_solve(payload: dict) -> str:
    variables = payload.get('variables') or ['x']
    vars_ = [sp.symbols(str(v)) for v in variables]
    local = {str(v): sym for v, sym in zip(variables, vars_)}
    equations = payload.get('equations') or []
    exprs = []
    for eq in equations:
        eq_text = str(eq)
        if '==' in eq_text:
            left, right = eq_text.split('==', 1)
        elif '=' in eq_text:
            left, right = eq_text.split('=', 1)
        else:
            left, right = eq_text, '0'
        exprs.append(sp.Eq(parse_math_expr(left, local), parse_math_expr(right, local)))
    target = str(payload.get('target', '')).lower()
    if not exprs: return 'No equation was provided.'
    sol = sp.solve(exprs, vars_[0] if len(vars_) == 1 else vars_, dict=False)
    result = f'solutions={sol}'
    if len(vars_) == 1 and isinstance(sol, list) and len(sol) == 2:
        r1, r2 = sol[0], sol[1]
        if 'positive_difference' in target or 'difference' in target:
            result += f'; positive_difference={sp.simplify(abs(r1-r2))}; approx={float(N(abs(r1-r2))):.8g}'
        if 'sum' in target: result += f'; sum={sp.simplify(r1+r2)}'
        if 'product' in target: result += f'; product={sp.simplify(r1*r2)}'
    return result

def run_sympy_simplify(payload: dict) -> str:
    try:
        latex = str(payload.get('latex', '')).strip()
        expr_text = str(payload.get('expression', '')).strip()
        if latex:
            from latex2sympy2 import latex2sympy
            expr = latex2sympy(latex)
        elif expr_text:
            expr = parse_math_expr(expr_text)
        else:
            return 'No expression or latex was provided.'
        simplified = sp.radsimp(sp.simplify(expr))
        return f'expr={expr}; simplified={simplified}; approx={float(N(simplified)):.8g}'
    except Exception as e:
        return f'sympy_simplify failed: {type(e).__name__}: {e}'

def run_sympy_calculus(payload: dict) -> str:
    op = str(payload.get('operation', '')).lower()
    var = sp.symbols(str(payload.get('variable', 'x')))
    expr = parse_math_expr(payload.get('expression', '0'), {str(var): var})
    if op == 'derivative': return f'derivative={sp.simplify(sp.diff(expr, var))}'
    if op == 'integral': return f'integral={sp.simplify(sp.integrate(expr, var))}'
    if op == 'limit':
        point = parse_math_expr(payload.get('point', '0'), {str(var): var})
        direction = payload.get('direction', '+-')
        return f'limit={sp.limit(expr, var, point, dir=direction)}'
    if op == 'critical_points':
        xs, ys = sp.symbols('x y')
        crit = sp.solve([sp.diff(expr, xs), sp.diff(expr, ys)], [xs, ys], dict=True)
        return f'critical_points={crit}'
    return 'Unknown calculus operation.'

def run_sympy_matrix(payload: dict) -> str:
    op = str(payload.get('operation', '')).lower()
    var_name = str(payload.get('variable', 'l'))
    l = sp.symbols(var_name)
    if op in ['char_poly', 'trace_det_eigen']:
        poly = parse_math_expr(payload.get('polynomial', '0'), {var_name: l})
        roots = sp.solve(sp.Eq(poly, 0), l)
        return f'eigenvalues={roots}; trace=sum_eigenvalues={sp.simplify(sum(roots)) if roots else None}; det=p(0)={sp.simplify(poly.subs(l, 0))}'
    return 'Unknown matrix operation.'

def run_probability_stats(payload: dict) -> str:
    op = str(payload.get('operation', '')).lower()
    v = payload.get('values') or {}
    try:
        if op == 'expected_value':
            outcomes = v.get('outcomes') or []
            ev = sum(float(o.get('prob', 0)) * float(o.get('value', 0)) for o in outcomes)
            return f'expected_value={ev}'
        if op == 'binomial':
            n, p = int(v['n']), float(v['p']); mode = v.get('mode', 'exactly'); k = int(v['k'])
            rng = [k] if mode == 'exactly' else range(k, n+1) if mode == 'at_least' else range(0, k+1)
            prob = sum(math.comb(n, r)*(p**r)*((1-p)**(n-r)) for r in rng)
            return f'binomial_probability={prob}; fraction={Fraction(prob).limit_denominator()}'
        if op in ['hypergeometric', 'without_replacement']:
            population = int(v['population'])
            success_population = int(v['success_population'])
            draws = int(v['draws'])
            successes = int(v['successes'])
            prob = (math.comb(success_population, successes) * math.comb(population - success_population, draws - successes)) / math.comb(population, draws)
            return f'hypergeometric_probability={prob}; fraction={Fraction(prob).limit_denominator()}'
        if op == 'bayes':
            prior, hit, false_pos = float(v['prior']), float(v['hit']), float(v['false_positive'])
            post = hit*prior/(hit*prior + false_pos*(1-prior))
            return f'bayes_posterior={post}'
        if op == 'normal_quartile_sigma':
            mean, value, z = float(v['mean']), float(v['value']), float(v.get('z', -0.67448975))
            return f'sigma={abs((value-mean)/z)}'
        if op == 'correlation_r2':
            r1, r2 = float(v['r1']), float(v['r2'])
            return f'r_squared_ratio={(r1*r1)/(r2*r2)}'
        if op == 'combinations':
            n, k = int(v['n']), int(v['k'])
            return f'combination={math.comb(n, k)}'
        if op == 'broken_stick_triangle':
            return 'broken_stick_triangle_probability=1/4=0.25=25%'
    except Exception as e:
        return f'probability_stats failed: {type(e).__name__}: {e}'
    return 'Unknown probability/statistics operation.'

def run_tool_call(tool_name: str, payload: dict) -> str:
    try:
        if tool_name == 'sympy_simplify': return run_sympy_simplify(payload)
        if tool_name == 'sympy_solve': return run_sympy_solve(payload)
        if tool_name == 'sympy_calculus': return run_sympy_calculus(payload)
        if tool_name == 'sympy_matrix': return run_sympy_matrix(payload)
        if tool_name == 'probability_stats': return run_probability_stats(payload)
        return f'Unknown tool: {tool_name}'
    except Exception as e:
        return f'Tool {tool_name} failed: {type(e).__name__}: {e}'

def normalize_tool_plan(plan):
    if not isinstance(plan, dict): return None
    clean = {str(k).strip().lower().replace(' ', '_'): v for k, v in plan.items()}
    action = str(clean.get('action', '')).strip().lower()
    tool = str(clean.get('tool', clean.get('tool_name', ''))).strip().lower()
    tool_aliases = {
        'sympy.simplify': 'sympy_simplify', 'sympy_simplifier': 'sympy_simplify', 'simplify': 'sympy_simplify',
        'sympy.solve': 'sympy_solve', 'sympy_solver': 'sympy_solve', 'solve': 'sympy_solve',
        'sympy.calculus': 'sympy_calculus', 'calculus': 'sympy_calculus', 'sympy.diff': 'sympy_calculus',
        'sympy.matrix': 'sympy_matrix', 'matrix': 'sympy_matrix',
        'probability.stats': 'probability_stats', 'probability': 'probability_stats', 'stats': 'probability_stats'
    }
    tool = tool_aliases.get(tool, tool)
    payload = clean.get('input') or {}
    return {'action': action, 'tool': tool, 'input': payload}

def try_llm_tool_solver(question, tokenizer, model, options_text: str):
    start = time.time()
    route_tokenizer = globals().get('planner_tokenizer', tokenizer)
    route_model = globals().get('planner_model', model)
    planner_messages = [
        {'role': 'system', 'content': 'You are a tool router. Think briefly, then write FINAL_JSON with one valid JSON object. Do not solve the problem or choose an option.'},
        {'role': 'user', 'content': f'{TOOL_DESCRIPTIONS}\nREAL QUESTION:\n{question.text}\n\nREAL OPTIONS:\n{options_text}\n\nThink briefly, then return FINAL_JSON.'}
    ]
    plan_raw = llm_chat(route_tokenizer, route_model, planner_messages, max_new_tokens=220, max_time=5.0)
    plan = normalize_tool_plan(parse_tool_plan(plan_raw))
    print(f'Tool plan raw {plan_raw.strip()}')
    valid_tools = ['sympy_simplify', 'sympy_solve', 'sympy_calculus', 'sympy_matrix', 'probability_stats']
    if not plan or plan.get('action') != 'tool' or plan.get('tool') not in valid_tools:
        print('Tool plan no valid tool used')
        return None
    tool_name = plan.get('tool')
    tool_input = plan.get('input') or {}
    tool_result = run_tool_call(tool_name, tool_input)
    print(f'Tool used {tool_name} input {tool_input} result {tool_result}')

    remaining = 28.5 - (time.time() - start)
    if remaining <= 1.0:
        return None
    final_messages = [
        {'role': 'system', 'content': 'Use the calculator result to choose the matching multiple-choice option. Be brief. Conclude with exactly one boxed letter.'},
        {'role': 'user', 'content': f'Question:\n{question.text}\n\nOptions:\n{options_text}\n\nCalculator result:\n{tool_result}\n\nReply with one final option: \\boxed{{A}}, \\boxed{{B}}, \\boxed{{C}}, or \\boxed{{D}}.'}
    ]
    final_raw = llm_chat(tokenizer, model, final_messages, max_new_tokens=120, max_time=remaining)
    letter, rule = extract_option_letter(final_raw)
    wrapup_used = False
    if rule == 'fallback':
        remaining = 28.5 - (time.time() - start)
        if remaining > 0.5:
            wrap_raw = llm_chat(tokenizer, model, final_messages + [{'role': 'assistant', 'content': final_raw}, {'role': 'user', 'content': 'Time is almost finished. Stop and submit nearest guess only as one boxed letter.'}], max_new_tokens=20, max_time=remaining)
            w_letter, w_rule = extract_option_letter(wrap_raw)
            final_raw += '\n' + wrap_raw
            letter, rule = w_letter, 'wrapup_' + w_rule
            wrapup_used = True
            print(f'Tool wrapup raw {wrap_raw.strip()}')
    print(f'Tool final raw\n{final_raw.strip()}')
    print(f'Tool extract letter {letter} rule {rule} wrapup {wrapup_used} elapsed {time.time()-start:.2f} seconds')
    idx = ['A', 'B', 'C', 'D'].index(letter) if letter in ['A', 'B', 'C', 'D'] else 2
    return question.options[idx].id, ['A', 'B', 'C', 'D'][idx], f'tool_{tool_name}_{rule}'


In [ ]:
def choose_answer_math_fixed(question, tokenizer, model) -> tuple:
    answer_start = time.time()
    options_text = "\n".join(f"{chr(65+i)}) {opt.text}" for i, opt in enumerate(question.options))
  
    if globals().get('USE_TOOL', False):
        tool_answer = try_llm_tool_solver(question, tokenizer, model, options_text)
        if tool_answer:
            return tool_answer
    else:
        print('Tool plan disabled because use tool is false')

    # Direct LLM fallback with token and time limits.
    messages = [
        {"role": "system", "content": "You are a math solver. Reason step-by-step but BE EXTREMELY BRIEF. DO NOT write long paragraphs. Conclude with your final option letter inside a box, like \\boxed{A}, \\boxed{B}, \\boxed{C}, or \\boxed{D}."},
        {"role": "user", "content": f"{question.text}\n\nOptions:\n{options_text}"}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    input_len = inputs['input_ids'].shape[1]

    THINK_TOKENS = 480
    WRAPUP_TOKENS = 20
    TOKEN_LIMIT = THINK_TOKENS + WRAPUP_TOKENS
    TIME_LIMIT = 27.0
    WRAPUP_TIME = 24.0

    stopping = TimeLimitStoppingCriteria(start_time=answer_start, time_limit=TIME_LIMIT)

    think_ids = model.generate(
        **inputs,
        max_new_tokens=THINK_TOKENS,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
        stopping_criteria=StoppingCriteriaList([stopping]),
    )

    think_new_tokens = think_ids.shape[1] - input_len
    think_text = tokenizer.decode(think_ids[0][input_len:], skip_special_tokens=True).strip()
    elapsed = time.time() - answer_start

    need_wrapup = False
    wrap_reason = 'none'
    if elapsed >= WRAPUP_TIME:
        need_wrapup = True
        wrap_reason = 'time'
    elif TOKEN_LIMIT - think_new_tokens <= WRAPUP_TOKENS:
        need_wrapup = True
        wrap_reason = 'tokens'
    elif not extract_answer_letter(think_text):
        need_wrapup = True
        wrap_reason = 'missing_answer'

    final_text = think_text
    if need_wrapup and elapsed < TIME_LIMIT:
        remaining_time = max(0.5, TIME_LIMIT - elapsed)
        wrap_messages = [
            {"role": "system", "content": "You must answer now. Use the previous work, make the nearest guess if unsure, and output only the final option letter in the form \\boxed{A}, \\boxed{B}, \\boxed{C}, or \\boxed{D}."},
            {"role": "user", "content": f"Question:\n{question.text}\n\nOptions:\n{options_text}\n\nPrevious work:\n{think_text[-1200:]}"}
        ]
        wrap_prompt = tokenizer.apply_chat_template(wrap_messages, tokenize=False, add_generation_prompt=True)
        wrap_inputs = tokenizer(wrap_prompt, return_tensors='pt').to(model.device)
        wrap_input_len = wrap_inputs['input_ids'].shape[1]
        wrap_stopping = TimeLimitStoppingCriteria(start_time=time.time(), time_limit=remaining_time)
        wrap_ids = model.generate(
            **wrap_inputs,
            max_new_tokens=WRAPUP_TOKENS,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
            stopping_criteria=StoppingCriteriaList([wrap_stopping]),
        )
        wrap_text = tokenizer.decode(wrap_ids[0][wrap_input_len:], skip_special_tokens=True).strip()
        final_text = think_text + "\n" + wrap_text

    print(f"Monitor think tokens {think_new_tokens} of {TOKEN_LIMIT} direct budget {TIME_LIMIT - elapsed:.2f} seconds think time {elapsed:.2f} seconds total time {time.time() - answer_start:.2f} seconds wrapup {need_wrapup} reason {wrap_reason}")
    if think_text:
        print("Model tail")
        print(think_text[-1200:])
    print("Model log")
    print(final_text)
    print("End log")

    # Extract final option letter.
    boxed = re.findall(r'\\boxed\{\s*([A-D])\s*\}', final_text, flags=re.I)
    if boxed:
        letter = boxed[-1].upper()
        idx = ord(letter) - 65
        print(f"Extract rule boxed letter {letter}")
        return question.options[idx].id, letter, 'boxed_latex_algebra'

    # First fallback: text such as option X or answer X.
    matches = re.findall(r'(?i)\b(?:option|answer|choice)\s*[:\-]?\s*([A-D])\b', final_text)
    if matches:
        letter = matches[-1].upper()
        idx = ord(letter) - 65
        print(f"Extract rule option word letter {letter}")
        return question.options[idx].id, letter, 'option_word'

    # Final fallback: last standalone A to D letter.
    matches = re.findall(r'\b([A-D])\b', final_text)
    if matches:
        letter = matches[-1].upper()
        idx = ord(letter) - 65
        print(f"Extract rule standalone letter {letter}")
        return question.options[idx].id, letter, 'fallback_letter'

    print('Could not extract answer; defaulting to A')
    return question.options[0].id, 'A', 'fallback_default'


In [ ]:
from millionaire_client import MillionaireClient
from millionaire_client.exceptions import TimeoutError, RateLimitError

client = MillionaireClient('http://131.175.15.22:51111/')
user   = client.login('gary', '13790229')
print(f"Logged in as {user.username}")

competitions = client.competitions.list_all()
# Assumes index 3 is the Maths competition. Change this if the competition order changes.
COMPETITION_ID = competitions[3].id

game = client.game.start(competition_id=COMPETITION_ID, mode='text')

while game.in_progress:
    question = game.current_question
    if question is None:
        break

    print(f"New question level {game.current_level} question {question.text}")
    for i, opt in enumerate(question.options):
        print(f"Option {chr(65+i)} {opt.text}")

    t0 = time.time()
    option_id, letter, method = choose_answer_math_fixed(question, tokenizer, model)
    t1 = time.time()

    print(f"Predicted {letter} Method {method} Time taken {t1-t0:.2f} seconds")

    try:
        result = game.answer(option_id)
        print(f"Correct {result.correct} Earned {result.earned_amount}")
    except TimeoutError:
        print("Timed out generation took more than thirty seconds")
        break
    except RateLimitError:
        print("Rate limited waiting five seconds")
        time.sleep(5)
        result = game.answer(option_id)
        print(f"Correct {result.correct} Earned {result.earned_amount}")

    if result.game_over:
        break
    time.sleep(1)

print(f"Game over final score {game.earned_amount}")
